# IELTS Task 2 band predictor — try it on your own essay

A Qwen2.5-0.5B-Instruct model fine-tuned with LoRA to return an overall band as strict JSON.
Repo: https://github.com/ahmedvictor507/ielts-essay-grader-lora

**Before you run:** turn **Internet ON** in the notebook settings (Settings -> Internet). A GPU is optional; CPU works for a 0.5B model.

**Read this first:** it predicts an *overall band only*, was trained on model-generated labels, and is a moderate ranking signal (Pearson r ~ 0.5), **not** a calibrated or official score. It tends to predict between 6 and 7 and rarely reaches the extremes.

In [ ]:
# 1) dependencies: the model was saved with transformers 5.x, so make sure we have it
!pip install -q -U "transformers>=5.0"

In [ ]:
# 2) get the code and the model weights (GitHub Release asset, ~1 GB)
!git clone -q https://github.com/ahmedvictor507/ielts-essay-grader-lora.git
%cd ielts-essay-grader-lora
!mkdir -p model-0.5B
BASE = "https://github.com/ahmedvictor507/ielts-essay-grader-lora/releases/download/v0.1"
for f in ["config.json", "generation_config.json", "tokenizer.json", "tokenizer_config.json", "chat_template.jinja", "model.safetensors"]:
    !wget -q -nc -P model-0.5B {BASE}/{f}
!ls -la model-0.5B

In [ ]:
# 3) load
from grader import load, grade
model, tok = load("model-0.5B")

In [ ]:
# 4) bundled demo essays (synthetic, written for the demo at three quality levels)
from pathlib import Path
prompt = Path("examples/prompt.txt").read_text().strip()
print("PROMPT:", prompt, "\n")
for name in ("weak", "mid", "strong"):
    r = grade(model, tok, prompt, Path(f"examples/essay_{name}.txt").read_text())
    print(f"essay_{name:<6}", r["raw"], f"({r['words']} words, {r['seconds']}s)")

In [ ]:
# 5) YOUR essay: paste a Task 2 question and essay between the triple quotes, then run this cell
PROMPT = """Paste the IELTS Task 2 question here."""

ESSAY = """Paste the essay here (the model was trained on essays of 100-600 words)."""

r = grade(model, tok, PROMPT, ESSAY)
print(r["raw"])
if r["warning"]:
    print("WARNING:", r["warning"])